In [8]:
import copy
import torch
from torch.optim import AdamW

from tqdm import tqdm

from src.mlconfgen.egnn import EGNNDynamics
from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from src.mlconfgen.utils import CONTEXT_NORMS
from src.mlconfgen.utils.mol_utils import prepare_edm_input

device = "mps"
TEACHER_STEPS = 10
BATCH = 2

# --- teacher ---
dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=420, device=device)
teacher = EquivariantDiffusion(dynamics=dyn, in_node_nf=8, timesteps=1000, noise_precision=1e-5)
ckpt = torch.load("edm_moi_chembl_15_39.pt", map_location=device)
teacher.load_state_dict(ckpt["state_dict"])

# --- adjust denosiing steps ---
teacher.gamma = PredefinedNoiseSchedule(timesteps=TEACHER_STEPS, precision=1e-5)
teacher.time_steps = torch.flip(torch.arange(TEACHER_STEPS, device=device), [0])
teacher.T = TEACHER_STEPS
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

norms = {k: torch.tensor(v, device=device) for k, v in ckpt.get("context_norms", CONTEXT_NORMS).items()}

# --- student (warm-start) ---
# student_dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=420, device=device)
# student_dyn.load_state_dict(copy.deepcopy(teacher.dynamics.state_dict()))

# --- student (cold-start) ---
student_dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=128, device=device)
# student_dyn.load_state_dict(copy.deepcopy(teacher.dynamics.state_dict()))

student = EquivariantFlowMatching(dynamics=student_dyn, in_node_nf=8).to(device)
opt = AdamW(student.parameters(), lr=1e-5)

# We will need to sample MOIs
ctx = torch.tensor([115.3558349609375, 307.8001708984375, 374.77020263671875], device=device)  # example

# --- train ---
for step in tqdm(range(5000)):
    node_mask, edge_mask, context = prepare_edm_input(
        BATCH, ctx, norms, min_n_nodes=26, max_n_nodes=26, device=device
    )
    with torch.inference_mode():
        z_T = teacher.sample_combined_position_feature_noise(BATCH, node_mask.size(1), node_mask)
        x1 = teacher.teach(z_T, node_mask, edge_mask, context)

    loss = student.compute_loss(z_T, x1, node_mask, edge_mask, context)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % 20 == 0:
        print(step, loss.item())

  0%|                   | 1/5000 [00:02<3:28:49,  2.51s/it]

0 25.008028030395508


  0%|                  | 21/5000 [00:28<1:48:55,  1.31s/it]

20 24.96201515197754


  1%|▏                 | 41/5000 [00:54<1:50:00,  1.33s/it]

40 24.86941909790039


  1%|▏                 | 61/5000 [01:21<1:47:59,  1.31s/it]

60 23.968820571899414


  1%|▏                 | 63/5000 [01:24<1:49:54,  1.34s/it]


KeyboardInterrupt: 